### 사전학습 모델 이용
- 사전학습 모델에 감성분류(긍정/중립/부정) 학습 및 결과 확인

### 데이터 불려오기

In [114]:
# !pip install konlpy
# from konlpy.tag import Okt

import pandas as pd
import numpy as np
import time
from tqdm import tqdm, tqdm_notebook
import re

def re_text(x):
    text = re.sub(r"[^ㄱ-ㅣ가-힣\s]", "", x)
    text = text.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","")
    text = text.replace('  는',"")
    return text

In [116]:
import gc
gc.collect()

0

In [117]:
df_train = pd.read_parquet('df_train.parquet')
df_train['gb'] = 'train'
df_test = pd.read_parquet('df_test.parquet')
df_test['gb'] = 'test'
df_val = pd.read_parquet('df_val.parquet')
df_val['gb'] = 'val'

df = pd.concat([df_test, df_train, df_val])
df = df.drop_duplicates(subset=['review'])
df.review = df.review.map(re_text)
df.shape, df.gb.value_counts()

((64008, 15),
 gb
 train    50950
 test     13058
 Name: count, dtype: int64)

In [31]:
# !pip install datasets

### 데이터 형태 맞추기

In [118]:
from datasets import load_dataset, Dataset, load_from_disk, DatasetDict
from json import loads, dumps

df_train.target_2 = df_train['target_2'].astype('int')
df_test.target_2 = df_test['target_2'].astype('int')
df_val.target_2 = df_val['target_2'].astype('int')

df_train_join  = df_train[['review', 'target_2']].rename({'target_2':'label'},axis=1).to_json(orient="records")
df_test_join  = df_test[['review',  'target_2']].rename({'target_2':'label'},axis=1).to_json(orient="records")
df_val_join  = df_val[['review',  'target_2']].rename({'target_2':'label'},axis=1).to_json(orient="records")

train_dataset = Dataset.from_list(loads(df_train_join))
val_dataset = Dataset.from_list(loads(df_val_join))
test_dataset = Dataset.from_list(loads(df_test_join))

class_dataset = DatasetDict({'train' : train_dataset,
                    'valid': val_dataset,
                    'test' : test_dataset})
class_dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 54757
    })
    valid: Dataset({
        features: ['review', 'label'],
        num_rows: 16428
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 13690
    })
})

In [119]:
class_dataset.save_to_disk('./data/')
# class_dataset = load_from_disk('./data/')

Saving the dataset (0/1 shards):   0%|          | 0/54757 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16428 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13690 [00:00<?, ? examples/s]

In [120]:
print(class_dataset['train'][1118])
print(class_dataset['train'][24515])

{'review': '지루할틈없이 계속 웃으며 달달하게 재밌게 봤습니다. 근래 제일 재밌게 봤네요 작품성 이런거 필요없이 진짜 재밌게 봤습니다!', 'label': 2}
{'review': '역시 가이리치감독님 직품답게 긴장감있고 좋네요 ㅎㅎ', 'label': 2}


In [40]:
class_dataset['train']['review'][:5]

['엘리멘탈 최고에요. 거의 2년정도 기다렸는데기대한것 보다 더 좋았어요. 원소의 디테일도 미쳤는데스토리가 더 대박이에요.',
 '너무 감동 받았어요',
 '배우와 캐릭터가 잘 맞물려서 더 코믹했음. 배우 캐스팅이 한 몫 한 듯 ㅇㅇ 가볍게 보기 좋았습니다~',
 '주연배우 연기 대단했다 진짜.. 강아지들은 너무 귀엽고~!',
 '여운이 많이 남았습니다.']

In [58]:
test_data = class_dataset['valid'].shuffle(seed=424)[:300]
td = pd.DataFrame(test_data)
td.head(2)

,review,labels
0,노래가 좋음.,2
1,솔직히 뭔소린지 모르겠다,1


### 모델확인

In [11]:
# TF_ENABLE_ONEDNN_OPTS=1
# 오류발생 I tensorflow/core/util/port.cc:113] oneDNN custom operations are on.  ~~
# !pip install tensorflow-intel
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf

In [103]:
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import pandas as pd
import numpy as np
import evaluate

pipe = pipeline('text-classification', model="matthewburke/korean_sentiment")

In [75]:
# !pip install evaluate

In [ ]:
preds = pipe(td['review'].tolist())

# 예측
preds_df = pd.DataFrame(preds)

In [59]:
preds_df.rename(columns={'label':'pred'}, inplace=True)
preds_df['pred'] = preds_df['pred'].map({'LABEL_2': 2, 'LABEL_1': 1, 'LABEL_0': 0})
preds_df = pd.concat([preds_df, td], axis=1)
preds_df.head(2)

,pred,score,review,target_2
0,1,0.968496,노래가 좋음.,2
1,0,0.959211,솔직히 뭔소린지 모르겠다,1


In [99]:
print(class_dataset['train'][3118])
print(class_dataset['train'][14310])

{'review': '좋은 영화 재밌게 관람했어요.', 'label': '2'}
{'review': '귀여워요', 'label': '2'}


In [77]:
### 토큰화
model_name = "kykim/bert-kor-base"
# model_name = "kykim/electra-kor-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [121]:
def tokenizer_func(x):
    return tokenizer(
        x['review'],
        # padding="max_length",
        max_length=256,
        truncation=True,
        padding=True
    )

tokenized_datasets = class_dataset.map(tokenizer_func, batched=True)

Map:   0%|          | 0/54757 [00:00<?, ? examples/s]

Map:   0%|          | 0/16428 [00:00<?, ? examples/s]

Map:   0%|          | 0/13690 [00:00<?, ? examples/s]

In [122]:
train_num_samples = 10000

train_ds = tokenized_datasets['train'].shuffle(seed=42).select(range(train_num_samples))
eval_ds = tokenized_datasets['valid'].shuffle(seed=42)

In [123]:
train_ds

Dataset({
    features: ['review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})

### 전이학습 Transfer Learning

In [76]:
model_name = "kykim/bert-kor-base"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at kykim/bert-kor-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# !pip install accelerate -U
!pip install -qq accelerate -U
# 설치 후 restart

In [129]:
bs = 64
epochs = 30
lr = 1e-5

args = TrainingArguments(
    'outputs',
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    evaluation_strategy='epoch',
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs,
    gradient_accumulation_steps=4, # until bs=128
    eval_accumulation_steps=4,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none',
    
)

In [130]:
metric = evaluate.load('accuracy')

# all Transformers models return logits
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

# def compute_metrics(eval_pred):
#     x, y = eval_preds
#     preds = np.argmax(x, -1)
#     return metric.compute(predictions=preds, references=y)

#### 학습

In [131]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [127]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
0,No log,0.684190,0.699598
1,No log,0.640446,0.726443
2,No log,0.630219,0.731130
3,No log,0.625222,0.733991


TrainOutput(global_step=312, training_loss=0.6827213580791767, metrics={'train_runtime': 362.1991, 'train_samples_per_second': 110.436, 'train_steps_per_second': 0.861, 'total_flos': 5247534003167232.0, 'train_loss': 0.6827213580791767, 'epoch': 3.987220447284345})

In [132]:
trainer.train()  # 에포크 증가

Epoch,Training Loss,Validation Loss,Accuracy
0,No log,0.625158,0.734539
1,No log,0.623944,0.738191
2,No log,0.621806,0.734052
4,No log,0.638756,0.745556
5,No log,0.634682,0.748661
6,No log,0.642281,0.745617
8,No log,0.689201,0.752739
9,No log,0.682445,0.752800
10,No log,0.702687,0.751583
12,0.443300,0.745821,0.753165


TrainOutput(global_step=1170, training_loss=0.2892881034785866, metrics={'train_runtime': 2578.0305, 'train_samples_per_second': 116.368, 'train_steps_per_second': 0.454, 'total_flos': 3.922073850020659e+16, 'train_loss': 0.2892881034785866, 'epoch': 29.808917197452228})

In [133]:
trainer.save_model("./mymodel")

#### 추론


In [144]:
pipe = pipeline('text-classification', model="./mymodel")

test_data = class_dataset['test'].shuffle(seed=424)[:1000]
td = pd.DataFrame(test_data)
td.head(2)

,review,label
0,현실과 환상의 모호한 경계,1
1,때려부시는 거 보려고 이런 영화 보는거죠.,1


In [145]:
preds = pipe(td['review'].tolist())

preds_df = pd.DataFrame(preds)
preds_df.rename(columns={'label':'pred'}, inplace=True)
preds_df['pred'] = preds_df['pred'].map({'LABEL_2': 2,'LABEL_1': 1, 'LABEL_0': 0})

In [146]:
preds_txts_df = pd.concat([preds_df, td], axis=1)
preds_txts_df.head(2)

,pred,score,review,label
0,1,0.666260,현실과 환상의 모호한 경계,1
1,1,0.968351,때려부시는 거 보려고 이런 영화 보는거죠.,1


In [147]:
preds_txts_df.pred.value_counts()

pred
2    679
1    232
0     89
Name: count, dtype: int64

In [148]:
preds_txts_df.label.value_counts()

label
2    657
1    237
0    106
Name: count, dtype: int64

In [149]:
mask = preds_txts_df['pred'] == preds_txts_df['label']

len(preds_txts_df[mask])

734

In [150]:
preds_txts_df[~mask]

,pred,score,review,label
2,2,0.986368,괴물은 누구나 될 수 있다.,1
4,1,0.920691,시원한 카레이싱과 액션 잘 봤어요,2
6,1,0.614486,2023 .03.29 개봉,2
10,1,0.993642,스토리에 딱히 집중은 안한거 같지만 아무래도 악어가 노래 부르는데 그건 중요한게 아닌듯,2
20,0,0.996913,예매는 하였지만 보지 않아서 돈은 날렸으나 시간을 아꼈다.,1
...,...,...,...,...
978,1,0.838385,최근 한국영화의 의외의 수확,2
982,2,0.984235,재밌네요,1
991,2,0.987800,바지사장계 에이스가 선보이는 추적극,1
994,2,0.950979,지니의 착한 활용법,1
